# LAB | Setting Up MCP in Codex

Connect Codex to the LangChain documentation MCP server, then validate that the assistant can use it.

In [1]:
!pip install langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv -q

In [ ]:
import os
from datetime import timedelta
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=20, max_retries=0)

# Configure the LangChain documentation MCP server up front so later cells can reuse it
mcp_client = MultiServerMCPClient({
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp"
        #"timeout": timedelta(seconds=10) - the query was hanging, therefore removed the timeout to see if it was a connectivity issue with the docs server
    }
})

print("MCP client configured. If tool loading hangs, the docs server is likely unreachable from this notebook environment.")

c:\Users\dilia\miniconda3\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


MCP client configured. If tool loading hangs, the docs server is likely unreachable from this notebook environment.


Loading MCP Tools into LangChain


In [2]:
# Load tools from the MCP server
# The client is stateless: get_tools() creates ephemeral sessions under the hood
try:
    mcp_tools = await mcp_client.get_tools()
except Exception as e:
    print(f"Failed to load MCP tools: {type(e).__name__}: {e}")
    raise

print(f"Loaded {len(mcp_tools)} tools from MCP server(s)")
print("\nAvailable tools:")
for tool in mcp_tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")

Loaded 2 tools from MCP server(s)

Available tools:
  - search_docs_by_lang_chain: Search across the Docs by LangChain knowledge base to find relevant information,...
  - query_docs_filesystem_docs_by_lang_chain: Run a read-only shell-like query against a virtualized, in-memory filesystem roo...


Running tests to check what might cause the query to stall

In [10]:
test = await mcp_tools[0].ainvoke({"query": "What is the latest version of LangChain?"})

In [11]:
test

[{'type': 'text',
  'text': "Title: What's new in LangChain v1\nLink: https://docs.langchain.com/oss/javascript/releases/langchain-v1#whats-new-in-langchain-v1\nPage: oss/javascript/releases/langchain-v1\nContent: What's new in <mark><b>LangChain</b></mark> v1\n<mark><b>LangChain</b></mark> v1 is a ",
  'id': 'lc_28e6b061-6912-4be7-bb3f-c2623ab731d2'},
 {'type': 'text',
  'text': "Title: What's new in LangChain v1\nLink: https://docs.langchain.com/oss/python/releases/langchain-v1#whats-new-in-langchain-v1\nPage: oss/python/releases/langchain-v1\nContent: What's new in <mark><b>LangChain</b></mark> v1\n<mark><b>LangChain</b></mark> v1 is a ",
  'id': 'lc_eb50b685-b1c9-42ea-a209-c779d3f30970'},
 {'type': 'text',
  'text': 'Title: LangChain overview\nLink: https://docs.langchain.com/oss/javascript/langchain/overview\nPage: oss/javascript/langchain/overview\nContent: <mark><b>LangChain</b></mark> overview\n<mark><b>LangChain</b></mark> is an open ',
  'id': 'lc_a37b0475-5576-4793-9b90-ad85

Create Agent

In [3]:
from langchain.agents import create_agent

# Reuse the tools loaded earlier instead of fetching them again
tools = mcp_tools

# Create agent with the model and MCP tools
# Use the 'prompt' parameter (not 'state_modifier', which is deprecated)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can answer questions about LangChain, LangGraph, and LangSmith by searching the official documentation. Always provide accurate, up-to-date information based on the documentation."
)

print(f"Agent created with {len(tools)} MCP tools")
print(f"Tools: {[tool.name for tool in tools]}")

Agent created with 2 MCP tools
Tools: ['search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain']


The error was in calling the llm. Had to be promptvalue

"message": "Invalid input type <class 'langchain_core.messages.human.HumanMessage'>. Must be a PromptValue, str, or list of BaseMessages.",

In [13]:
llm.invoke([{"role": "user", "content": "What is LCEL (LangChain Expression Language)?"}])

AIMessage(content="LangChain Expression Language (LCEL) is a specialized language designed for constructing and manipulating expressions within the LangChain framework, which is primarily used for building applications that leverage large language models (LLMs). LCEL allows developers to create complex queries and expressions that can interact with various components of the LangChain ecosystem, such as data retrieval, processing, and model interaction.\n\nKey features of LCEL include:\n\n1. **Simplicity**: LCEL is designed to be easy to read and write, making it accessible for developers who may not have extensive experience with programming languages.\n\n2. **Integration**: It seamlessly integrates with LangChain's components, allowing for efficient data handling and model interaction.\n\n3. **Flexibility**: LCEL can be used to express a wide range of operations, from simple queries to more complex data manipulations, enabling developers to build sophisticated applications.\n\n4. **Ex

Querying LangChain Documentation

In [14]:
# Query the agent about LangChain
question = "What is LCEL (LangChain Expression Language)?"
print(f"Question: {question}\n")

result = await agent.ainvoke({
    "messages": [{"role": "user", "content": question}]
})

print(f"Answer: {result['messages'][-1].content}")

Question: What is LCEL (LangChain Expression Language)?

Answer: The **LangChain Expression Language (LCEL)** is a powerful feature within the LangChain framework that allows users to define and manipulate expressions in a structured way. While specific details about LCEL are not extensively documented in the current LangChain resources, it is generally used to facilitate the creation of complex queries and operations within LangChain applications.

### Key Features of LCEL:
- **Structured Expressions**: LCEL enables users to create structured expressions that can be evaluated and manipulated programmatically.
- **Integration with LangChain**: It is designed to work seamlessly with other components of the LangChain framework, enhancing the capabilities of applications built using LangChain.
- **Flexibility**: Users can define custom expressions that suit their specific needs, making it a versatile tool for developers.

### Usage:
LCEL can be utilized in various contexts within LangChai